<a href="https://colab.research.google.com/github/Damian200211/PharmaRoute/blob/main/document_Q%26A_bot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y llama-cpp-python
!pip install --no-cache-dir --only-binary llama-cpp-python llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!pip install -q PyPDF2 langchain-text-splitters llama-index llama-index-embeddings-huggingface gradio
!pip install llama-index-llms-llama-cpp
!pip install "uvicorn<0.30.0"

Found existing installation: llama_cpp_python 0.3.34
Uninstalling llama_cpp_python-0.3.34:
  Successfully uninstalled llama_cpp_python-0.3.34
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 44.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import gradio as gr
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode

In [ ]:
# ==========================================
# 1. SETUP & CONFIGURATION (MISTRAL GGUF)
# ==========================================
print("Downloading and Loading Mistral 7B Q4_K_M (takes ~1 min on GPU)...")

Settings.llm = LlamaCPP(
    model_url="https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    model_path=None,
    temperature=0.1,
    max_new_tokens=256,
    context_window=4096,
    model_kwargs={"n_gpu_layers": -1}, # -1 offloads ALL layers to the GPU
    verbose=False,
)

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# 2. HELPER FUNCTIONS (CLASSIFICATION)
# ==========================================
VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

def local_llm_predict(prompt):
    """Sends prompt to Mistral using required [INST] formatting."""
    # Fixed the duplicate <s> warning here!
    formatted_prompt = f"[INST] {prompt} [/INST]"
    response = Settings.llm.complete(formatted_prompt)
    return str(response.text)

def clean_doc_type(response):
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return cleaned_title

def classify_document_type(text):
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the page
    content below, classify it into ONE of these document types:
    Cover Letter, Certificate of Quality, Packaging Specification, BSE/TSE Declaration, Material Description, Supplier Qualification, Chain of Custody, Other.

    Page Content: {text[:1000]}

    Respond with ONLY the document type name. No explanation.
    """
    return clean_doc_type(local_llm_predict(prompt))

def is_same_document(prev_text, curr_text, doc_type=None):
    prompt = f"""
    Do these two pages belong to the SAME pharmaceutical document?
    Previous page type: {doc_type or 'unknown'}
    Previous Page (snippet): {prev_text[:500]}
    Current Page (snippet): {curr_text[:500]}
    Answer ONLY 'Yes' or 'No'.
    """
    response = local_llm_predict(prompt)
    return response.strip().lower().startswith("yes")

def predict_doc_type_for_query(query):
    prompt = f"""
    Which pharmaceutical document type is most likely to answer this query: "{query}"?
    Choose ONLY ONE from: Cover Letter, Certificate of Quality, Packaging Specification, BSE/TSE Declaration, Material Description, Supplier Qualification, Chain of Custody, Other.
    Respond with ONLY the document type name. No explanation.
    """
    return clean_doc_type(local_llm_predict(prompt))

In [ ]:
# ==========================================
# 3. GRADIO PIPELINE LOGIC (DRAG & DROP)
# ==========================================
def build_index_from_pdf(pdf_file):
    """Parses, classifies, chunks, and indexes a drag-and-dropped PDF."""
    if pdf_file is None:
        return None, " Error: Please upload a PDF file first."

    file_path = pdf_file.name
    file_name = os.path.basename(file_path)

    reader = PdfReader(file_path)
    doc_pages = [{"page_num": i, "text": page.extract_text() or ""} for i, page in enumerate(reader.pages)]

    metadata_store = []
    current_doc_type = None
    page_counter = 0

    for i, page in enumerate(doc_pages):
        if i == 0:
            current_doc_type = classify_document_type(page["text"])
        else:
            prev_text = doc_pages[i - 1]["text"]
            same = is_same_document(prev_text, page["text"], current_doc_type)
            if not same:
                current_doc_type = classify_document_type(page["text"])
                page_counter = 0
            else:
                page_counter += 1

        metadata_store.append({
            "page": i,
            "text": page["text"],
            "doc_type": current_doc_type,
            "page_in_doc": page_counter
        })

    logical_docs = []
    current_doc = {"text": "", "doc_type": None, "page_start": 0}

    for page in metadata_store:
        if page["page_in_doc"] == 0 and current_doc["text"]:
            current_doc["page_end"] = page["page"] - 1
            logical_docs.append(current_doc)
            current_doc = {"text": "", "doc_type": None, "page_start": page["page"]}

        current_doc["text"] += "\n\n" + page["text"]
        current_doc["doc_type"] = page["doc_type"]

    current_doc["page_end"] = metadata_store[-1]["page"]
    logical_docs.append(current_doc)

    splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
    all_documents = []

    for doc_id, logical_doc in enumerate(logical_docs):
        chunks = splitter.split_text(logical_doc["text"])
        for chunk_idx, chunk in enumerate(chunks):
            all_documents.append(
                Document(
                    text=chunk,
                    metadata={
                        "doc_type": logical_doc["doc_type"],
                        "chunk_index": chunk_idx,
                        "doc_id": f"doc_{doc_id}",
                        "page_start": logical_doc["page_start"],
                        "page_end": logical_doc["page_end"],
                        "source_file": file_name
                    }
                )
            )

    index = VectorStoreIndex.from_documents(all_documents)
    status_msg = f" Successfully processed '{file_name}'!\n• Pages: {len(reader.pages)}\n• Logical Documents: {len(logical_docs)}\n• Chunks Indexed: {len(all_documents)}"

    return index, status_msg

def process_query(user_query, index):
    """Routes the query, retrieves chunks, and synthesizes an answer."""
    if index is None:
        return " Please upload and process a PDF document first!", "N/A", "N/A"
    if not user_query.strip():
        return "Please enter a question.", "N/A", "N/A"

    predicted_doc_type = predict_doc_type_for_query(user_query)

    retriever = index.as_retriever(
        similarity_top_k=4,
        filters=MetadataFilters(filters=[
            MetadataFilter(key="doc_type", value=predicted_doc_type, operator=FilterOperator.EQ)
        ])
    )
    retrieved_nodes = retriever.retrieve(user_query)

    if not retrieved_nodes:
        return f"No matching chunks found under category '{predicted_doc_type}'.", predicted_doc_type, "N/A"

    synthesizer = get_response_synthesizer(response_mode=ResponseMode.COMPACT)
    response = synthesizer.synthesize(user_query, nodes=retrieved_nodes)

    chunks_text = ""
    for idx, node in enumerate(retrieved_nodes):
        meta = node.metadata
        chunks_text += f"--- Chunk {idx + 1} (Pages {meta.get('page_start')}-{meta.get('page_end')}) ---\n"
        chunks_text += f"{node.text[:250]}...\n\n"

    return str(response.response), predicted_doc_type, chunks_text

In [ ]:
with gr.Blocks(title="Dynamic AI Document Intelligence") as demo:
    gr.Markdown("# AI Document Intelligence & Query Router")
    gr.Markdown("Drag and drop any pharmaceutical/contract PDF blob below. The local Mistral AI will automatically classify, chunk, and index it for metadata-routed retrieval.")

    # State variable to hold the vector index in memory
    vector_index_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="Drag & Drop PDF Here",
                file_types=[".pdf"],
                file_count="single"
            )
            process_btn = gr.Button(" Process & Index PDF", variant="primary")
            status_output = gr.Textbox(label="Indexing Status", lines=5, interactive=False)

        with gr.Column(scale=2):
            user_query = gr.Textbox(
                lines=2,
                placeholder="e.g., Were there any packaging configuration changes?",
                label="Ask a Question"
            )
            query_btn = gr.Button(" Query Document", variant="secondary")

            ai_answer = gr.Textbox(label="AI Answer", lines=4, interactive=False)
            doc_route = gr.Textbox(label="Predicted Document Route", interactive=False)
            retrieved_context = gr.Textbox(label="Retrieved Context (First 250 chars per chunk)", lines=6, interactive=False)

    # Event Handlers
    process_btn.click(
        fn=build_index_from_pdf,
        inputs=[pdf_input],
        outputs=[vector_index_state, status_output]
    )

    query_btn.click(
        fn=process_query,
        inputs=[user_query, vector_index_state],
        outputs=[ai_answer, doc_route, retrieved_context]
    )

print("Launching Gradio Interface...")
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Launching Gradio Interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95866b4e3216c6b49c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
